# Regime Detection and Regime Label Table Creation

This notebook reproduces the changepoint-based regime detection workflow used for the study.

It covers the following steps:

1. load weekly market series from BigQuery,
2. standardize them for changepoint detection,
3. run a penalty sensitivity check,
4. generate changepoints with the selected penalty,
5. create the `regime_labels` table in BigQuery.

## 0. Before you start

Authenticate locally before running BigQuery-related cells.

```bash
gcloud auth login
gcloud config set project <YOUR_GCP_PROJECT_ID>
gcloud auth application-default login
```

In [ ]:
import json
from pathlib import Path

import bigframes.pandas as bpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ruptures as rpt
from google.cloud import bigquery

# --- Configuration ---
PROJECT_ID = "<YOUR_GCP_PROJECT_ID>"
BASE_DATASET_ID = "<YOUR_BASE_BIGQUERY_DATASET>"          # e.g. "base"
ANALYTICS_DATASET_ID = "<YOUR_ANALYTICS_BIGQUERY_DATASET>"  # e.g. "analytics"

SOURCE_TABLE_NFT_TRADING_USD_PREFILTER = "nft_trading_usd_prefilter"
SOURCE_TABLE_USD_ETH_BASE = "usd_eth_base"
TARGET_TABLE_REGIME_LABELS = "regime_labels"

PENALTY_GRID = (4, 6, 8, 10)
SELECTED_PENALTY = 6
CP_MERGE_TOL_DAYS = 84
MIN_GLOBAL_CP_DATE = "2018-01-01"

OUTPUT_DIR = Path("../outputs/regime_detection")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = bigquery.Client(project=PROJECT_ID)


def fqn(dataset_id: str, table_name: str) -> str:
    return f"{PROJECT_ID}.{dataset_id}.{table_name}"


NFT_TRADING_USD_PREFILTER_TABLE = fqn(BASE_DATASET_ID, SOURCE_TABLE_NFT_TRADING_USD_PREFILTER)
USD_ETH_BASE_TABLE = fqn(BASE_DATASET_ID, SOURCE_TABLE_USD_ETH_BASE)
REGIME_LABELS_TABLE = fqn(ANALYTICS_DATASET_ID, TARGET_TABLE_REGIME_LABELS)

## 1. Weekly series loaders

In [ ]:
def to_pandas_weekly(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["week_start"] = pd.to_datetime(out["week_start"]).dt.tz_localize(None)
    return out


def prep_weekly_series(df: pd.DataFrame, series_name: str) -> pd.DataFrame:
    out = df.copy()
    out["week_start"] = pd.to_datetime(out["week_start"])
    out["value"] = out["value"].astype(float).fillna(0.0)
    out["series"] = series_name
    out["value_log"] = np.log1p(out["value"])
    return out[["week_start", "value", "value_log", "series"]]


def load_weekly_traders() -> pd.DataFrame:
    sql = f"""
    SELECT
      week_start,
      COUNT(DISTINCT wallet) AS value
    FROM (
        SELECT week_start, buyer AS wallet
        FROM `{NFT_TRADING_USD_PREFILTER_TABLE}`
        WHERE buyer IS NOT NULL

        UNION ALL

        SELECT week_start, seller AS wallet
        FROM `{NFT_TRADING_USD_PREFILTER_TABLE}`
        WHERE seller IS NOT NULL
    )
    GROUP BY week_start
    ORDER BY week_start
    """
    return to_pandas_weekly(bpd.read_gbq(sql).to_pandas())


def load_weekly_collections() -> pd.DataFrame:
    sql = f"""
    SELECT
      week_start,
      COUNT(DISTINCT collection) AS value
    FROM `{NFT_TRADING_USD_PREFILTER_TABLE}`
    WHERE collection IS NOT NULL
    GROUP BY week_start
    ORDER BY week_start
    """
    return to_pandas_weekly(bpd.read_gbq(sql).to_pandas())


def load_weekly_volumes() -> pd.DataFrame:
    sql = f"""
    SELECT
      week_start,
      CAST(SUM(price_usd) AS FLOAT64) AS value
    FROM `{NFT_TRADING_USD_PREFILTER_TABLE}`
    GROUP BY week_start
    ORDER BY week_start
    """
    return to_pandas_weekly(bpd.read_gbq(sql).to_pandas())


def load_weekly_median_price() -> pd.DataFrame:
    sql = f"""
    SELECT
      week_start,
      APPROX_QUANTILES(price_usd, 100)[OFFSET(50)] AS value
    FROM `{NFT_TRADING_USD_PREFILTER_TABLE}`
    GROUP BY week_start
    ORDER BY week_start
    """
    return to_pandas_weekly(bpd.read_gbq(sql).to_pandas())


def load_weekly_usd_eth_rate() -> pd.DataFrame:
    sql = f"""
    SELECT
      week_start,
      AVG(usd_eth_rate) AS value
    FROM `{USD_ETH_BASE_TABLE}`
    GROUP BY week_start
    ORDER BY week_start
    """
    return to_pandas_weekly(bpd.read_gbq(sql).to_pandas())


def build_weekly_series_dataframe() -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    df_trader = load_weekly_traders()
    df_coll = load_weekly_collections()
    df_vol = load_weekly_volumes()
    df_price = load_weekly_median_price()
    df_eth = load_weekly_usd_eth_rate()

    df_wallets = prep_weekly_series(df_trader, "Wallets")
    df_collections = prep_weekly_series(df_coll, "Collections")
    df_volume = prep_weekly_series(df_vol, "Market Volume")
    df_price_series = prep_weekly_series(df_price, "Median Market Price")
    df_eth_series = prep_weekly_series(df_eth, "ETH/USD")

    df_all = pd.concat(
        [df_wallets, df_collections, df_volume, df_price_series, df_eth_series],
        ignore_index=True,
    )

    df_dict = {
        "wallets": df_wallets,
        "collections": df_collections,
        "volume": df_volume,
        "price": df_price_series,
        "eth": df_eth_series,
    }

    return df_all, df_dict


df_all, df_dict = build_weekly_series_dataframe()
print("df_all shape:", df_all.shape)
print(df_all["series"].unique())
df_all.head()

## 2. Changepoint detection helpers

In [ ]:
def detect_cp(series: pd.Series, pen: int = 6) -> list[int]:
    values = (
        series.astype(float)
        .ffill()
        .fillna(0.0)
        .values.reshape(-1, 1)
    )
    algo = rpt.Pelt(model="rbf").fit(values)
    idxs = algo.predict(pen=pen)
    return idxs[:-1]


def merge_cp_by_date(cp_dates_dict: dict[str, list[pd.Timestamp]], tol_days: int = 84) -> list[pd.Timestamp]:
    all_dates = sorted({d for dates in cp_dates_dict.values() for d in dates})
    merged = []
    for d in all_dates:
        if not merged or abs((d - merged[-1]).days) > tol_days:
            merged.append(d)
    return merged


def vote_score(date: pd.Timestamp, cp_dates: dict[str, list[pd.Timestamp]], radius_days: int = 84) -> int:
    score = 0
    for dates in cp_dates.values():
        if any(abs((date - d).days) <= radius_days for d in dates):
            score += 1
    return score


def auto_regime_boundaries(
    df_dict: dict[str, pd.DataFrame],
    tol_days: int = 84,
    pen: int = 6,
    min_global_cp_date: str = MIN_GLOBAL_CP_DATE,
) -> pd.DataFrame:
    cp_dates: dict[str, list[pd.Timestamp]] = {}
    series_labels: dict[str, str] = {}

    for key, df in df_dict.items():
        df2 = df.sort_values("week_start").reset_index(drop=True)
        if df2.empty:
            cp_dates[key] = []
            series_labels[key] = key
            continue

        series_label = df2["series"].iloc[0]
        series_labels[key] = series_label
        idxs = detect_cp(df2["value_log"], pen=pen)
        cp_dates[key] = [pd.Timestamp(df2["week_start"].iloc[i]) for i in idxs]

    cp_merged = merge_cp_by_date(cp_dates, tol_days)
    scored = [(d, vote_score(d, cp_dates, tol_days)) for d in cp_merged]
    scored = sorted(scored, key=lambda x: (-x[1], x[0]))

    final_cps = []
    for d, _ in scored:
        if not final_cps or abs((d - final_cps[-1]).days) > tol_days:
            final_cps.append(d)

    cp_points_global = [
        d for d in final_cps if d >= pd.Timestamp(min_global_cp_date)
    ]

    boom_start = next((d for d in cp_points_global if d.year == 2021), None)
    post_start = next((d for d in cp_points_global if d.year == 2022), None)

    rows = []
    for key, dates in cp_dates.items():
        series_label = series_labels.get(key, key)
        for d in dates:
            rows.append(
                {
                    "date": d,
                    "cp_type": "local",
                    "series_key": key,
                    "series": series_label,
                    "is_global": 0,
                    "score": vote_score(d, cp_dates, tol_days),
                    "is_boom_start": 1 if (boom_start is not None and d == boom_start) else 0,
                    "is_post_start": 1 if (post_start is not None and d == post_start) else 0,
                }
            )

    for d in cp_points_global:
        rows.append(
            {
                "date": d,
                "cp_type": "global",
                "series_key": "all",
                "series": "ALL",
                "is_global": 1,
                "score": vote_score(d, cp_dates, tol_days),
                "is_boom_start": 1 if (boom_start is not None and d == boom_start) else 0,
                "is_post_start": 1 if (post_start is not None and d == post_start) else 0,
            }
        )

    cp_df = pd.DataFrame(rows)
    if cp_df.empty:
        return pd.DataFrame(
            columns=[
                "date",
                "cp_type",
                "series_key",
                "series",
                "is_global",
                "score",
                "is_boom_start",
                "is_post_start",
            ]
        )

    return cp_df.sort_values("date").reset_index(drop=True)


def auto_regime_boundary_dates(
    df_dict: dict[str, pd.DataFrame],
    tol_days: int = 84,
    pen: int = 6,
    min_global_cp_date: str = MIN_GLOBAL_CP_DATE,
) -> tuple[pd.Timestamp | None, pd.Timestamp | None, list[pd.Timestamp]]:
    cp_df = auto_regime_boundaries(
        df_dict=df_dict,
        tol_days=tol_days,
        pen=pen,
        min_global_cp_date=min_global_cp_date,
    )
    if cp_df.empty:
        return None, None, []

    cp_points_global = cp_df.loc[cp_df["is_global"] == 1, "date"].tolist()
    boom_candidates = cp_df.loc[cp_df["is_boom_start"] == 1, "date"]
    post_candidates = cp_df.loc[cp_df["is_post_start"] == 1, "date"]

    boom_start = boom_candidates.iloc[0] if not boom_candidates.empty else None
    post_start = post_candidates.iloc[0] if not post_candidates.empty else None
    return boom_start, post_start, cp_points_global


def make_pen_sensitivity_table(
    df_dict: dict[str, pd.DataFrame],
    pens: tuple[int, ...] = PENALTY_GRID,
    tol_days: int = CP_MERGE_TOL_DAYS,
) -> pd.DataFrame:
    rows = []
    for p in pens:
        boom_start, post_start, cps = auto_regime_boundary_dates(df_dict, tol_days=tol_days, pen=p)
        rows.append(
            {
                "pen": p,
                "boom_start": boom_start.strftime("%Y-%m-%d") if boom_start is not None else "NA",
                "post_start": post_start.strftime("%Y-%m-%d") if post_start is not None else "NA",
                "n_global_candidates": len(cps),
            }
        )
    return pd.DataFrame(rows)

## 3. Penalty sensitivity check

In [ ]:
pen_df = make_pen_sensitivity_table(
    df_dict=df_dict,
    pens=PENALTY_GRID,
    tol_days=CP_MERGE_TOL_DAYS,
)
pen_df

## 4. Generate changepoints with the selected penalty

In [ ]:
cp_df = auto_regime_boundaries(
    df_dict=df_dict,
    tol_days=CP_MERGE_TOL_DAYS,
    pen=SELECTED_PENALTY,
)
cp_df.head(20)

In [ ]:
if cp_df.empty:
    raise ValueError("No changepoints were detected. Review the selected penalty and source series.")

BOOM_START = cp_df.query("is_boom_start == 1")["date"].iloc[0]
POST_START = cp_df.query("is_post_start == 1")["date"].iloc[0]

print("BOOM_START:", BOOM_START)
print("POST_START:", POST_START)

## 5. Optional exports

In [ ]:
cp_df.to_csv(OUTPUT_DIR / "changepoints_pen_6.csv", index=False)
pen_df.to_csv(OUTPUT_DIR / "penalty_sensitivity.csv", index=False)

with open(OUTPUT_DIR / "selected_boundaries.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "selected_penalty": SELECTED_PENALTY,
            "boom_start": str(BOOM_START.date()),
            "post_start": str(POST_START.date()),
        },
        f,
        indent=2,
    )

## 6. Create the BigQuery `regime_labels` table

In [ ]:
def create_regime_labels_table(
    boom_start: pd.Timestamp,
    post_start: pd.Timestamp,
    overwrite: bool = True,
) -> None:
    ddl = f"""
    CREATE OR REPLACE TABLE `{REGIME_LABELS_TABLE}` AS
    WITH bounds AS (
      SELECT
        DATE '{boom_start.date()}' AS boom_start,
        DATE '{post_start.date()}' AS post_boom_start
    ),
    weekly_trades AS (
      SELECT DISTINCT week_start
      FROM `{NFT_TRADING_USD_PREFILTER_TABLE}`
    )
    SELECT
      week_start,
      CASE
        WHEN week_start < boom_start THEN 'pre-boom'
        WHEN week_start < post_boom_start THEN 'boom'
        ELSE 'post-boom'
      END AS regime_class,
      boom_start,
      post_boom_start,
      {SELECTED_PENALTY} AS selected_penalty
    FROM weekly_trades
    CROSS JOIN bounds
    ORDER BY week_start
    """
    client.query(ddl).result()


create_regime_labels_table(BOOM_START, POST_START)
print(f"Created: {REGIME_LABELS_TABLE}")

In [ ]:
query = f"SELECT * FROM `{REGIME_LABELS_TABLE}` ORDER BY week_start LIMIT 20"
regime_labels_preview = client.query(query).to_dataframe()
regime_labels_preview